[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/10_debugging_dojo.ipynb)

# ⚒️ Act II · Quest 10 — 🥋 The Debugging Dojo

*Seven broken training loops. Seven classic PyTorch bugs. Diagnose each symptom, fix the code, earn your belt.*

⬅️ [09_attention_build_a_gpt](09_attention_build_a_gpt.ipynb)  •  [11_art_of_creation](11_art_of_creation.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## Welcome to the Dojo 🥋

Everything below **runs without crashing** — that's what makes these bugs dangerous. The model
just silently underperforms. For each station:

1. Run the broken code. **Study the symptom.**
2. Form a diagnosis *before* touching anything.
3. Write your fix in the attempt cell and grade it with `check(...)`.
4. Only then read the sensei's solution at the bottom.

These seven bugs account for a shocking fraction of all real-world "my model won't train" posts.

In [ ]:
# Shared training arena: a simple blob-classification task
def make_blobs(n=600, seed=0):
    g = torch.Generator().manual_seed(seed)
    centers = torch.tensor([[-2., 0.], [2., 0.], [0., 2.5]])
    X = torch.cat([c + 0.7 * torch.randn(n // 3, 2, generator=g) for c in centers])
    y = torch.arange(3).repeat_interleave(n // 3)
    perm = torch.randperm(n, generator=g)
    return X[perm], y[perm]

Xb, yb = make_blobs()
def fresh_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 3))

def acc(model, X=Xb, y=yb):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

## 🥋 Station 1 — The loss that refuses to settle

In [ ]:
model = fresh_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05)
for step in range(200):
    loss = F.cross_entropy(model(Xb), yb)
    loss.backward()
    opt.step()                    # 🐛 something is missing from this loop...
    if step % 40 == 0:
        print(f"step {step:3d}: loss {loss.item():8.3f}")
print("accuracy:", f"{acc(model)*100:.0f}%", " <- symptom: loss bounces/explodes instead of settling")

**Diagnosis:** gradients *accumulate* across steps (Act I, Rule 1). Every step applies the sum of ALL past gradients. Fix it in the cell below.

In [ ]:
# ⚔️ your fix — rewrite the loop correctly, then: check("station1", model)

## 🥋 Station 2 — The loss that can't count

In [ ]:
model = fresh_model()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(300):
    opt.zero_grad()
    loss = F.mse_loss(model(Xb), yb.float().unsqueeze(1))   # 🐛 MSE... on class labels?!
    loss.backward(); opt.step()
print("accuracy:", f"{acc(model)*100:.0f}%", " <- symptom: loss falls but accuracy is garbage")

**Diagnosis:** class indices (0,1,2) are *categories*, not quantities — regressing onto them treats class 2 as 'twice class 1'. Classification wants `cross_entropy` on logits.

In [ ]:
# ⚔️ your fix here, then: check("station2", model)

## 🥋 Station 3 — The rocket ship

In [ ]:
model = fresh_model()
opt = torch.optim.SGD(model.parameters(), lr=25.0)   # 🐛 lr=25?!
for step in range(100):
    opt.zero_grad()
    loss = F.cross_entropy(model(Xb), yb)
    loss.backward(); opt.step()
    if step % 25 == 0:
        print(f"step {step:3d}: loss {loss.item():.2e}")
print("symptom: loss shoots to infinity / NaN — every step overshoots the valley")

In [ ]:
# ⚔️ your fix here, then: check("station3", model)

## 🥋 Station 4 — The silent broadcast

In [ ]:
# A regression task this time: predict y = 3x from noisy data
xr = torch.linspace(-2, 2, 200).unsqueeze(1)
yr = 3 * xr.squeeze() + 0.1 * torch.randn(200)        # 🐛 shape (200,) ... predictions are (200,1)

reg = nn.Linear(1, 1)
opt = torch.optim.Adam(reg.parameters(), lr=0.05)
for step in range(200):
    opt.zero_grad()
    loss = F.mse_loss(reg(xr).squeeze(-1) if False else reg(xr), yr)   # (200,1) vs (200,) -> (200,200)!!
    loss.backward(); opt.step()
print(f"learned weight: {reg.weight.item():.3f}  (should be ~3.0)")
print("symptom: loss 'trains' but the learned weight is wrong — broadcasting built a 200x200 error matrix")

**Diagnosis:** `(200,1) - (200,)` broadcasts to `(200,200)` — MSE against every *pair*. Newer PyTorch warns about this; older versions are silent. Make the shapes match.

In [ ]:
# ⚔️ your fix here — retrain reg so its weight is ~3, then: check("station4", reg)

## 🥋 Station 5 — The frozen layer

In [ ]:
class TwoStage(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Linear(2, 16)
        self.stage2 = nn.Linear(16, 3)
    def forward(self, x):
        h = torch.relu(self.stage1(x))
        h = torch.tensor(h.detach().numpy())     # 🐛 out of the graph, then back in
        return self.stage2(h)

model = TwoStage()
before = model.stage1.weight.clone()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(200):
    opt.zero_grad()
    F.cross_entropy(model(Xb), yb).backward()
    opt.step()
print("stage1 weights changed:", not torch.allclose(before, model.stage1.weight))
print("accuracy:", f"{acc(model)*100:.0f}%", " <- symptom: trains 'okay' but stage1 never learns — gradient is severed")

In [ ]:
# ⚔️ your fix — a TwoStage whose stage1 actually learns, then: check("station5", model)

## 🥋 Station 6 — The gambling evaluator

In [ ]:
drop_model = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 3))
opt = torch.optim.Adam(drop_model.parameters(), lr=1e-2)
for step in range(300):
    opt.zero_grad()
    F.cross_entropy(drop_model(Xb), yb).backward()
    opt.step()

# 🐛 evaluating without model.eval() — dropout is still firing!
with torch.no_grad():
    a1 = (drop_model(Xb).argmax(1) == yb).float().mean().item()
    a2 = (drop_model(Xb).argmax(1) == yb).float().mean().item()
print(f"eval #1: {a1*100:.1f}%   eval #2: {a2*100:.1f}%   <- symptom: same data, different answers?!")

In [ ]:
# ⚔️ your fix — evaluate properly and submit the (deterministic) accuracy: check("station6", accuracy_value)

## 🥋 Station 7 — The unshuffled deck

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Data arrives SORTED by class (very common in real datasets!)
order = torch.argsort(yb)
Xs, ys = Xb[order], yb[order]

model = fresh_model()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loader = DataLoader(TensorDataset(Xs, ys), batch_size=32, shuffle=False)   # 🐛 shuffle=False
for epoch in range(3):
    for xb2, yb2 in loader:
        opt.zero_grad()
        F.cross_entropy(model(xb2), yb2).backward()
        opt.step()
print("accuracy:", f"{acc(model)*100:.0f}%",
      " <- symptom: each epoch ends having seen ONLY class-2 batches last; the model keeps 'forgetting'")

In [ ]:
# ⚔️ your fix here, then: check("station7", model)

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_TH = 0.90
_register("station1", 15, lambda m: acc(m) >= _TH, "add opt.zero_grad() at the top of the loop")
_register("station2", 15, lambda m: acc(m) >= _TH, "use F.cross_entropy(model(Xb), yb) — logits + integer labels")
_register("station3", 15, lambda m: acc(m) >= _TH, "a sane learning rate: 0.01–0.1 for SGD here")
_register("station4", 15, lambda r: abs(r.weight.item() - 3.0) < 0.2, "match shapes: compare (200,1) with (200,1), or squeeze to (200,)")
_register("station5", 15, lambda m: acc(m) >= _TH and isinstance(m, nn.Module), "delete the numpy round-trip — keep h as the tensor it was")
_register("station6", 15, lambda a: isinstance(a, float) and a >= _TH, "model.eval() before inference (and torch.no_grad()); submit the accuracy float")
_register("station7", 15, lambda m: acc(m) >= _TH, "shuffle=True in the DataLoader")

---
## 🎓 Sensei's solutions

Scroll no further until you've earned your attempts. Each solution below runs and passes its check.

In [ ]:
# Station 1 — zero_grad
model = fresh_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05)
for step in range(200):
    opt.zero_grad()
    F.cross_entropy(model(Xb), yb).backward()
    opt.step()
check("station1", model)

In [ ]:
# Station 2 — right loss for the job
model = fresh_model()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(300):
    opt.zero_grad()
    F.cross_entropy(model(Xb), yb).backward()
    opt.step()
check("station2", model)

In [ ]:
# Station 3 — sane learning rate
model = fresh_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05)
for step in range(300):
    opt.zero_grad()
    F.cross_entropy(model(Xb), yb).backward()
    opt.step()
check("station3", model)

In [ ]:
# Station 4 — matching shapes
reg = nn.Linear(1, 1)
opt = torch.optim.Adam(reg.parameters(), lr=0.05)
yr_col = yr.unsqueeze(1)                      # (200,) -> (200,1)
for step in range(300):
    opt.zero_grad()
    F.mse_loss(reg(xr), yr_col).backward()
    opt.step()
check("station4", reg)

In [ ]:
# Station 5 — never leave the graph
class TwoStageFixed(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Linear(2, 16)
        self.stage2 = nn.Linear(16, 3)
    def forward(self, x):
        return self.stage2(torch.relu(self.stage1(x)))

model = TwoStageFixed()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(300):
    opt.zero_grad()
    F.cross_entropy(model(Xb), yb).backward()
    opt.step()
check("station5", model)

In [ ]:
# Station 6 — eval mode for evaluation
drop_model.eval()
with torch.no_grad():
    a = (drop_model(Xb).argmax(1) == yb).float().mean().item()
drop_model.train()
check("station6", a)

In [ ]:
# Station 7 — shuffle your data
model = fresh_model()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loader = DataLoader(TensorDataset(Xs, ys), batch_size=32, shuffle=True)
for epoch in range(4):
    for xb2, yb2 in loader:
        opt.zero_grad()
        F.cross_entropy(model(xb2), yb2).backward()
        opt.step()
check("station7", model)

xp_report()

---
## 🥋 Black belt earned

Commit the seven to memory — as *symptoms*, not just rules:

| Symptom | First suspect |
|---------|--------------|
| Loss bounces or climbs steadily | missing `zero_grad()` |
| Loss falls, accuracy is garbage | wrong loss for the task |
| Loss → `inf`/`NaN` in a few steps | learning rate too high |
| "Trains" but learns wrong values | silent broadcasting in the loss |
| One part of the model never improves | graph severed (`.detach()`, `.numpy()`, `.item()`) |
| Same input, different eval results | forgot `model.eval()` (dropout/batchnorm) |
| Loss cycles weirdly per epoch | unshuffled, class-sorted data |

**End of Act II.** Act III: creation (generative models), action (RL), and speed (deployment).